# Network Intrusion Detection on IoT Traffic (CICIoT2023)

CPS 536 Machine Learning, Fall 2026 — milestone code.

This notebook contains the whole pipeline:

1. **Setup** — imports and settings
2. **Data download** — 8 of the 63 merged CICIoT2023 CSV files (~1.2 GB, 5.7M rows)
3. **Data cleaning** — NaN/inf removal, de-duplication, conflicting labels, label mapping, per-label cap, stratified 70/15/15 split
4. **Models** — Logistic Regression (taught in class) and XGBoost (not taught in class)
5. **Experiments** — binary and 8-class tasks, with/without class weights, `IAT` ablation
6. **Results** — tables, per-class recall/F1, confusion matrices, feature importance

Every random step uses seed 42. The full run takes about 35–45 minutes on a 12-core laptop with 16 GB RAM, almost all of it XGBoost training.

## 1. Setup

In [ ]:
# !pip install pandas numpy pyarrow scikit-learn xgboost matplotlib
import glob
import json
import os
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix,
                             f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

SEED = 42
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
RESULTS_DIR = "results"
N_FILES = 8          # how many of the 63 merged CSVs to use
CAP = 100_000        # max rows kept per fine-grained label
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Data download

CICIoT2023 (Neto et al., *Sensors* 2023) from the Canadian Institute for Cybersecurity, UNB:
<https://www.unb.ca/cic/datasets/iotdataset-2023.html>. The official release is split into 63 "merged" CSVs, each a random mix of all classes.
We download the first 8 from a public Hugging Face mirror of that release. Files already on disk are skipped.

In [ ]:
BASE = "https://huggingface.co/datasets/bencorn/CIC-IoT-2023/resolve/main/CSV/MERGED_CSV"
for i in range(1, N_FILES + 1):
    name = f"Merged{i:02d}.csv"
    path = os.path.join(RAW_DIR, name)
    if not os.path.exists(path):
        print("downloading", name)
        subprocess.run(["curl", "-sSL", "--retry", "5", "--retry-all-errors",
                        "-o", path, f"{BASE}/{name}"], check=True)
for f in sorted(glob.glob(os.path.join(RAW_DIR, "*.csv"))):
    print(f"{os.path.basename(f)}  {os.path.getsize(f) / 1e6:.1f} MB")

## 3. Data cleaning

Steps:

1. Read each CSV as float32 to save memory, and normalise the label strings.
2. Drop rows with NaN / ±inf values.
3. Drop exact duplicate rows (features + label), first within each file and then across files. If duplicates are kept, identical rows land in both train and test.
4. Drop feature vectors that appear with **more than one label**, since no model can classify them.
5. Drop constant features (none in this data).
6. Map the 34 fine-grained labels to 8 classes (benign + 7 attack families) and a binary label.
7. Cap every fine-grained label at `CAP` rows. This trims the huge flood classes and keeps every rare class whole.
8. Stratified 70/15/15 split on the fine-grained label.

In [ ]:
FAMILY = {
    "BENIGN": "Benign",
    "DICTIONARYBRUTEFORCE": "BruteForce",
    "MITM-ARPSPOOFING": "Spoofing", "DNS_SPOOFING": "Spoofing",
    "RECON-HOSTDISCOVERY": "Recon", "RECON-OSSCAN": "Recon",
    "RECON-PORTSCAN": "Recon", "RECON-PINGSWEEP": "Recon",
    "VULNERABILITYSCAN": "Recon",
    "SQLINJECTION": "Web", "COMMANDINJECTION": "Web", "XSS": "Web",
    "BACKDOOR_MALWARE": "Web", "UPLOADING_ATTACK": "Web",
    "BROWSERHIJACKING": "Web",
}


def family(label):
    if label in FAMILY:
        return FAMILY[label]
    for prefix, fam in (("DDOS", "DDoS"), ("DOS", "DoS"), ("MIRAI", "Mirai")):
        if label.startswith(prefix):
            return fam
    raise ValueError(f"unknown label {label}")

In [ ]:
files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
stats = {"rows_raw": 0, "rows_nan_inf": 0, "rows_dup_within_file": 0}
parts = []
for f in files:
    df = pd.read_csv(f, engine="pyarrow")
    stats["rows_raw"] += len(df)
    feats = [c for c in df.columns if c != "Label"]
    df[feats] = df[feats].astype(np.float32)
    df["Label"] = df["Label"].str.strip().str.upper()

    bad = ~np.isfinite(df[feats].to_numpy()).all(axis=1)
    stats["rows_nan_inf"] += int(bad.sum())
    df = df[~bad]

    n = len(df)
    df = df.drop_duplicates()
    stats["rows_dup_within_file"] += n - len(df)
    parts.append(df)
    print(f"{os.path.basename(f)}: {len(df):,} rows after within-file cleaning")

df = pd.concat(parts, ignore_index=True)
del parts
n = len(df)
df = df.drop_duplicates(ignore_index=True)
stats["rows_dup_across_files"] = n - len(df)

feats = [c for c in df.columns if c != "Label"]
ambiguous = df.duplicated(subset=feats, keep=False)
stats["rows_ambiguous_label"] = int(ambiguous.sum())
ambiguous_counts = df.loc[ambiguous, "Label"].value_counts()
df = df[~ambiguous].reset_index(drop=True)

constant = [c for c in feats if df[c].nunique() <= 1]
df = df.drop(columns=constant)
FEATURES = [c for c in feats if c not in constant]
stats["constant_features_dropped"] = constant
stats["n_features"] = len(FEATURES)
stats["rows_clean"] = len(df)
pd.Series(stats)

**Conflicting labels.** The rows dropped in step 4 are almost all DoS and DDoS versions of the same flood: the same feature vector appears with both labels. This places a ceiling on how well any model can separate DoS from DDoS using these features.

In [ ]:
ambiguous_counts.head(10).to_frame('rows with a conflicting duplicate')

In [ ]:
rng = np.random.default_rng(SEED)
keep = df.groupby("Label")["Label"].transform(
    lambda s: pd.Series(rng.permutation(len(s)) < CAP, index=s.index))
df = df.loc[keep].reset_index(drop=True)
df["Family"] = df["Label"].map(family)
df["Binary"] = (df["Family"] != "Benign").astype(np.int8)

train_df, rest = train_test_split(df, test_size=0.30, stratify=df["Label"],
                                  random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=0.50, stratify=rest["Label"],
                                   random_state=SEED)
del rest
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_parquet(os.path.join(PROCESSED_DIR, f"{name}.parquet"), index=False)

stats.update(rows_sampled=len(df), rows_train=len(train_df),
             rows_val=len(val_df), rows_test=len(test_df))
with open(os.path.join(PROCESSED_DIR, "cleaning_stats.json"), "w") as fh:
    json.dump(stats, fh, indent=2)

counts = df["Family"].value_counts()
display(pd.DataFrame({"rows": counts, "share %": (100 * counts / len(df)).round(2)}))
print(f"train {len(train_df):,}  val {len(val_df):,}  test {len(test_df):,}")

## 4. Models

**Logistic Regression.** A linear model with a sigmoid output (binary) or softmax output (8 classes), trained with cross-entropy plus an L2 penalty of strength `C`. Many features are heavy-tailed, so we apply a signed log transform `sign(x)·log(1+|x|)` and then standardise. Both are fitted on training data only. `C` ∈ {0.01, 0.1, 1, 10} is tuned on validation macro-F1.

**XGBoost.** Gradient-boosted decision trees: each new tree fits the gradient and Hessian of the loss of the current ensemble, and leaf weights are L2-regularised. Settings: histogram splits, learning rate 0.1, subsampling 0.8, column subsampling 0.8. The number of trees is chosen by early stopping on validation log-loss (patience 30), and `max_depth` ∈ {6, 10} is tuned on validation macro-F1.

**Class imbalance.** By default both models use balanced class weights (each class contributes equally to the loss). We also run both without weights.

**Evaluation.** Hyper-parameters are chosen on validation; all reported numbers come from the untouched test set.

In [ ]:
FAMILIES = ["Benign", "DDoS", "DoS", "Mirai", "Recon", "Spoofing", "Web", "BruteForce"]


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


def make_data(task, drop=()):
    feats = [c for c in FEATURES if c not in drop]
    if task == "binary":
        names = ["Benign", "Attack"]
        enc = lambda d: d["Binary"].to_numpy()
    else:
        names = FAMILIES
        idx = {n: i for i, n in enumerate(FAMILIES)}
        enc = lambda d: d["Family"].map(idx).to_numpy()
    data = {s: (d[feats].to_numpy(np.float32), enc(d))
            for s, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
    return data, feats, names


def make_lr(C, weighted):
    return make_pipeline(
        FunctionTransformer(signed_log1p),
        StandardScaler(),
        LogisticRegression(C=C, max_iter=2000, tol=1e-4,
                           class_weight="balanced" if weighted else None))


def make_xgb(depth, n_classes):
    params = dict(n_estimators=1500, learning_rate=0.1, max_depth=depth,
                  subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                  early_stopping_rounds=30, n_jobs=-1, random_state=SEED)
    if n_classes == 2:
        params.update(objective="binary:logistic", eval_metric="logloss")
    else:
        params.update(objective="multi:softprob", eval_metric="mlogloss")
    return XGBClassifier(**params)


def fit(model_name, hp, data, n_classes, weighted):
    Xtr, ytr = data["train"]
    Xva, yva = data["val"]
    if model_name == "lr":
        m = make_lr(hp, weighted)
        m.fit(Xtr, ytr)
        return m
    m = make_xgb(hp, n_classes)
    sw = compute_sample_weight("balanced", ytr) if weighted else None
    sw_va = [compute_sample_weight("balanced", yva)] if weighted else None
    m.fit(Xtr, ytr, sample_weight=sw, eval_set=[(Xva, yva)],
          sample_weight_eval_set=sw_va, verbose=False)
    return m


def evaluate(m, X, y, names):
    proba = m.predict_proba(X)
    pred = proba.argmax(axis=1)
    if len(names) == 2:
        auc = roc_auc_score(y, proba[:, 1])
    else:
        auc = roc_auc_score(y, proba, multi_class="ovr", average="macro")
    return pred, {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
        "roc_auc": auc,
    }

In [ ]:
def plot_confusion(y, pred, names, title, path):
    cm = confusion_matrix(y, pred, labels=range(len(names)))
    cmn = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(1.0 + 0.8 * len(names), 0.6 + 0.7 * len(names)))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(names)), names, rotation=45, ha="right")
    ax.set_yticks(range(len(names)), names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center",
                    fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True (row-normalised = recall)")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return cm.tolist()


def plot_importance(m, feats, title, path, top=15):
    imp = m.get_booster().get_score(importance_type="gain")
    s = pd.Series({feats[int(k[1:])]: v for k, v in imp.items()})
    s = (s / s.sum()).sort_values().tail(top)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.barh(s.index, s.values, color="#3b6ea5")
    ax.set_xlabel("Share of total gain")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return s.sort_values(ascending=False).to_dict()


def run(model_name, task, weighted=True, drop=()):
    tag = f"{model_name}_{task}" + ("" if weighted else "_noweights")
    if drop:
        tag += "_drop-" + "-".join(drop)
    out = os.path.join(RESULTS_DIR, tag)
    os.makedirs(out, exist_ok=True)

    data, feats, names = make_data(task, drop)
    grid = [0.01, 0.1, 1.0, 10.0] if model_name == "lr" else [6, 10]
    tuning, best = [], None
    for hp in grid:
        t0 = time.time()
        m = fit(model_name, hp, data, len(names), weighted)
        _, val = evaluate(m, *data["val"], names)
        row = {"hp": hp, "val_macro_f1": round(val["macro_f1"], 4),
               "fit_seconds": round(time.time() - t0, 1)}
        if model_name == "xgb":
            row["trees"] = int(m.best_iteration) + 1
        tuning.append(row)
        print(tag, row)
        if best is None or val["macro_f1"] > best[1]:
            best = (m, val["macro_f1"], hp)
    m, _, best_hp = best

    pred, test = evaluate(m, *data["test"], names)
    yte = data["test"][1]
    result = {"tag": tag, "model": model_name, "task": task,
              "weighted": weighted, "dropped": list(drop), "best_hp": best_hp,
              "tuning": tuning, "test": test,
              "per_class": classification_report(
                  yte, pred, labels=range(len(names)), target_names=names,
                  digits=4, output_dict=True, zero_division=0),
              "confusion_matrix": plot_confusion(
                  yte, pred, names, tag, os.path.join(out, "confusion.png"))}
    if model_name == "xgb":
        result["importance"] = plot_importance(
            m, feats, f"{tag}: feature importance (gain)",
            os.path.join(out, "importance.png"))
    with open(os.path.join(out, "metrics.json"), "w") as fh:
        json.dump(result, fh, indent=2)
    print(tag, "TEST", {k: round(v, 4) for k, v in test.items()})
    return result

## 5. Experiments

| Run | Model | Task | Class weights | Features |
|---|---|---|---|---|
| `lr_binary` | Logistic Regression | benign vs attack | balanced | all 39 |
| `xgb_binary` | XGBoost | benign vs attack | balanced | all 39 |
| `lr_family` | Logistic Regression | 8-class | balanced | all 39 |
| `xgb_family` | XGBoost | 8-class | balanced | all 39 |
| `lr_family_noweights` | Logistic Regression | 8-class | none | all 39 |
| `xgb_family_noweights` | XGBoost | 8-class | none | all 39 |
| `xgb_family_drop-IAT` | XGBoost | 8-class | balanced | without `IAT` |

In [ ]:
EXPERIMENTS = [
    dict(model_name="lr", task="binary"),
    dict(model_name="xgb", task="binary"),
    dict(model_name="lr", task="family"),
    dict(model_name="xgb", task="family"),
    dict(model_name="lr", task="family", weighted=False),
    dict(model_name="xgb", task="family", weighted=False),
    dict(model_name="xgb", task="family", drop=("IAT",)),
]
results = {}
for exp in EXPERIMENTS:
    r = run(**exp)
    results[r["tag"]] = r

## 6. Results

### 6.1 Test-set metrics

In [ ]:
summary = pd.DataFrame({
    tag: {"best hp": r["best_hp"], **{k: round(v, 4) for k, v in r["test"].items()}}
    for tag, r in results.items()}).T
summary

### 6.2 Per-class recall / F1 (8-class task, test set)

In [ ]:
fam_runs = [t for t, r in results.items() if r["task"] == "family"]
per_class = pd.DataFrame({
    t: {c: f"{results[t]['per_class'][c]['recall']:.3f} / {results[t]['per_class'][c]['f1-score']:.3f}"
        for c in FAMILIES}
    for t in fam_runs})
per_class.insert(0, "test rows", [int(results[fam_runs[0]]["per_class"][c]["support"])
                                  for c in FAMILIES])
per_class

### 6.3 Confusion matrices

In [ ]:
def show(paths, ncols=2, size=7):
    nrows = -(-len(paths) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * 0.85 * nrows))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, p in zip(np.ravel(axes), paths):
        ax.imshow(plt.imread(p))
    plt.tight_layout()
    plt.show()

show([os.path.join(RESULTS_DIR, t, "confusion.png") for t in ["lr_binary", "xgb_binary"]], size=5)
show([os.path.join(RESULTS_DIR, t, "confusion.png") for t in fam_runs])

### 6.4 XGBoost feature importance

In [ ]:
show([os.path.join(RESULTS_DIR, t, "importance.png")
      for t in ["xgb_binary", "xgb_family", "xgb_family_noweights", "xgb_family_drop-IAT"]], size=6)

### 6.5 Observations

- With the same settings, **XGBoost beats Logistic Regression** on every metric in both tasks.
- **Mirai** is easy for both models (F1 ≈ 0.99–1.00).
- **DoS vs. DDoS** is the largest source of error. These are the same floods that produced identical feature vectors with conflicting labels during cleaning.
- **Benign, Recon, Spoofing, Web and Brute Force** are confused with one another. XGBoost separates them much better than the linear model.
- **Rare classes** (Web, Brute Force) reach only about 0.5 recall with balanced weights, and precision is low.
- **Class weights trade majority-class accuracy for minority-class recall.** Without weights, macro-F1 goes up, but Web and Brute Force recall collapses (to zero for Logistic Regression on Web).
- **Removing `IAT`** changes almost nothing. `Number` (packets per window) dominates XGBoost's gain, which deserves a check as a possible artefact of how the data was generated.
- **Scores are below published ones** (around 99%), likely because we remove the 42% duplicate rows before splitting and use only 8 of the 63 files.